In [1]:
import pandas as pd
import re

news = pd.read_csv("csv/all_news.csv")

KECAMATAN = {
    "Ampelgading": (-8.2583, 112.7667),
    "Bantur": (-8.3167, 112.5167),
    "Bululawang": (-8.1333, 112.6833),
    "Dampit": (-8.2333, 112.7833),
    "Dau": (-7.9833, 112.5667),
    "Donomulyo": (-8.2833, 112.4167),
    "Gedangan": (-8.3333, 112.6667),
    "Gondanglegi": (-8.1667, 112.6333),
    "Jabung": (-7.9667, 112.7167),
    "Kalipare": (-8.2167, 112.4667),
    "Karangploso": (-7.9333, 112.6167),
    "Kasembon": (-7.8167, 112.3667),
    "Kepanjen": (-8.1333, 112.5667),
    "Kromengan": (-8.1000, 112.5167),
    "Lawang": (-7.8333, 112.6833),
    "Ngajum": (-8.0833, 112.5000),
    "Ngantang": (-7.8833, 112.3833),
    "Pagak": (-8.2000, 112.5167),
    "Pagelaran": (-8.1500, 112.5833),
    "Pakis": (-7.9833, 112.7000),
    "Pakisaji": (-8.1000, 112.6000),
    "Poncokusumo": (-8.0167, 112.7667),
    "Pujon": (-7.9000, 112.4167),
    "Singosari": (-7.9000, 112.6667),
    "Sumbermanjing Wetan": (-8.3667, 112.6833),
    "Sumberpucung": (-8.1667, 112.5333),
    "Tajinan": (-8.1000, 112.6667),
    "Tirtoyudo": (-8.2833, 112.7167),
    "Tumpang": (-8.0000, 112.7333),
    "Turen": (-8.1667, 112.6833),
    "Wagir": (-8.0333, 112.5500),
    "Wajak": (-8.0833, 112.7167),
    "Wonosari": (-8.1500, 112.5167),
}

KECAMATAN_ALIASES = {
    "sumbermanjing": "Sumbermanjing Wetan",
    "sumbermanjingwetan": "Sumbermanjing Wetan",
    "smw": "Sumbermanjing Wetan",
    "kpj": "Kepanjen",
}

ALL_NAMES = {}

for k in KECAMATAN:
    ALL_NAMES[k.lower()] = k

for alias, official in KECAMATAN_ALIASES.items():
    ALL_NAMES[alias.lower()] = official


def detect_location(row):

    title = str(row["title"]).lower()
    content = str(row["content"]).lower()

    scores = {}

    for keyword, official in ALL_NAMES.items():

        pattern = rf"\b{re.escape(keyword)}\b"

        title_hits = len(re.findall(pattern, title))
        content_hits = len(re.findall(pattern, content))

        score = 0

        score += title_hits * 10
        score += content_hits * 1

        location_patterns = [
            rf"terjadi di.*?{re.escape(keyword)}",
            rf"berlokasi di.*?{re.escape(keyword)}",
            rf"di kecamatan.*?{re.escape(keyword)}",
            rf"wilayah.*?{re.escape(keyword)}",
            rf"desa .*?{re.escape(keyword)}",
        ]

        for p in location_patterns:
            if re.search(p, content):
                score += 20

        if score > 0:
            scores[official] = scores.get(official, 0) + score

    if not scores:
        return pd.Series([
            [],
            None,
            None,
            None
        ])

    scores = dict(
        sorted(
            scores.items(),
            key=lambda x: x[1],
            reverse=True
        )
    )

    all_kecamatan = list(scores.keys())

    primary = all_kecamatan[0]

    lat = KECAMATAN[primary][0]
    lon = KECAMATAN[primary][1]

    return pd.Series([
        all_kecamatan,
        primary,
        lat,
        lon
    ])


news[
    [
        "all_kecamatan",
        "primary_kecamatan",
        "latitude",
        "longitude"
    ]
] = news.apply(
    detect_location,
    axis=1
)

news.to_csv(
    "csv/all_news_with_kecamatan.csv",
    index=False
)

print(news.shape)

print(
    news["primary_kecamatan"]
    .value_counts()
)

print(
    news[
        [
            "title",
            "primary_kecamatan"
        ]
    ].head(20)
)

(108, 11)
primary_kecamatan
Bululawang      4
Kalipare        3
Lawang          3
Gondanglegi     2
Ampelgading     2
Sumberpucung    2
Bantur          2
Pagelaran       1
Kasembon        1
Wonosari        1
Pakis           1
Kepanjen        1
Karangploso     1
Singosari       1
Wajak           1
Dau             1
Name: count, dtype: int64
                                                title primary_kecamatan
0   Desa Sumberejo Gelar Semarak Selamatan ke-116,...         Pagelaran
1   Polres Batu Gelar Layanan On the Spot di CFD S...               NaN
2   Cegah Kebutaan Sejak Dini, Klinik Mata Batu Aj...               NaN
3   Satlantas Polres Batu Buka Layanan SIM dan Sam...               NaN
4   31 Anak Terpilih Jadi Anggota Pocil Polres Bat...               NaN
5   Sambut HUT Bhayangkara ke-80, Polres Batu Buka...               NaN
6   Kebakaran Pabrik di Junrejo Batu Dipadamkan Se...               NaN
7   Satresnarkoba Polres Batu Gencarkan Edukasi An...               NaN
8   Ngopi 

In [2]:
news["primary_kecamatan"].value_counts()

primary_kecamatan
Bululawang      4
Kalipare        3
Lawang          3
Gondanglegi     2
Ampelgading     2
Sumberpucung    2
Bantur          2
Pagelaran       1
Kasembon        1
Wonosari        1
Pakis           1
Kepanjen        1
Karangploso     1
Singosari       1
Wajak           1
Dau             1
Name: count, dtype: int64

In [3]:
news["primary_kecamatan"].isna().sum()

np.int64(81)

In [4]:
print(news["primary_kecamatan"].value_counts())

print(
    "NULL:",
    news["primary_kecamatan"].isna().sum()
)

print(
    "Coverage:",
    round(
        news["primary_kecamatan"].notna().mean()*100,
        2
    ),
    "%"
)

primary_kecamatan
Bululawang      4
Kalipare        3
Lawang          3
Gondanglegi     2
Ampelgading     2
Sumberpucung    2
Bantur          2
Pagelaran       1
Kasembon        1
Wonosari        1
Pakis           1
Kepanjen        1
Karangploso     1
Singosari       1
Wajak           1
Dau             1
Name: count, dtype: int64
NULL: 81
Coverage: 25.0 %


In [5]:
failed = news[
    news["primary_kecamatan"].isna()
]

print(
    failed[
        ["title", "url"]
    ]
)

                                                 title  \
1    Polres Batu Gelar Layanan On the Spot di CFD S...   
2    Cegah Kebutaan Sejak Dini, Klinik Mata Batu Aj...   
3    Satlantas Polres Batu Buka Layanan SIM dan Sam...   
4    31 Anak Terpilih Jadi Anggota Pocil Polres Bat...   
5    Sambut HUT Bhayangkara ke-80, Polres Batu Buka...   
..                                                 ...   
103   7.230 Pendaftar Jalur Domisili SDSMP Bakal Gugur   
104      Keluhkan Nilai TKA Tak Muncul di Website SPMB   
105  Dorong Sertifikasi Halal di Sekolah dan Madras...   
106  Prestasi Gemilang! SMKN 8 Kota Malang Raih Jua...   
107  Prestasi Membanggakan, 16 Siswa SMKN 8 Kota Ma...   

                                                   url  
1    https://beritamalang.media/polres-batu-gelar-l...  
2    https://beritamalang.media/cegah-kebutaan-seja...  
3    https://beritamalang.media/satlantas-polres-ba...  
4    https://beritamalang.media/31-anak-terpilih-ja...  
5    https://berit